# Método das linhas e bibliotecas

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/16-metodo-das-linhas.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. transformar uma EDP de evolução num sistema de EDOs, discretizando só o espaço (o **método das linhas**), e resolvê-lo com os integradores dos capítulos 12 e 13;
> 2. estimar os autovalores do sistema semidiscreto e explicar por que ele é rígido, com rigidez que cresce como $1/\Delta x^2$;
> 3. informar a um integrador implícito a jacobiana esparsa (ou só o seu padrão de esparsidade), e saber o que acontece quando isso não é feito;
> 4. separar e verificar os erros de espaço e de tempo;
> 5. entender a ideia dos elementos finitos (forma fraca, Galerkin, montagem) num problema 1D, e reconhecer as bibliotecas que fazem isso na prática.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
import time
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve
from scipy.integrate import solve_ivp, trapezoid
from cnlivro.edp import laplaciano_1d, laplaciano_2d, fem_1d

## 1. Problema de engenharia: o espalhador de calor de um chip

Um módulo eletrônico tem um chip de 1 cm × 1 cm que dissipa 100 W. Ele está soldado no centro de uma placa metálica de 5 cm × 5 cm, o **espalhador de calor**, cuja face de baixo está apoiada numa placa fria, com coeficiente de troca $h = 1000$ W/(m² K) para um fluido a 30 °C. As bordas laterais são isoladas. O espalhador distribui o calor do chip por uma área 25 vezes maior antes de entregá-lo ao fluido. A temperatura da placa sob o chip não pode passar de **90 °C**.

Como a placa é fina, a temperatura varia pouco na espessura $e$, e o balanço de energia num elemento $dx\,dy$ da placa dá a equação do calor 2D com fonte e perda:

$$
\rho c\,e\,\frac{\partial T}{\partial t} = k\,e\,\nabla^2 T + q''(x, y) - h\,(T - T_a),
$$ (eq-espalhador)

com $q'' = 100\ \text{W}/(1\ \text{cm}^2) = 10^6$ W/m² sob o chip e zero fora dele, e $\partial T/\partial n = 0$ nas bordas. Queremos duas coisas: o aquecimento nos primeiros 30 s depois de ligar o chip, e a escolha do material (cobre ou alumínio) e da espessura que dão a placa **mais leve** que respeita o limite.

In [ ]:
L_p, lado_chip = 0.05, 0.01               # m
P_chip, h_conv, T_a = 100.0, 1000.0, 30.0  # W, W/(m² K), °C
materiais = {"cobre": dict(k=390.0, rho=8900.0, c=385.0), "alumínio": dict(k=237.0, rho=2700.0, c=900.0)}

def monta_placa(n, k, rho, c, e):
    # volumes finitos n × n (centros das células); bordas isoladas
    dx = L_p / n
    xc = (np.arange(n) + 0.5) * dx
    X, Y = np.meshgrid(xc, xc, indexing="ij")
    chip = (np.abs(X - L_p / 2) < lado_chip / 2) & (np.abs(Y - L_p / 2) < lado_chip / 2)
    q = np.where(chip, P_chip / lado_chip**2, 0.0).ravel()            # W/m²
    rce = rho * c * e
    A = (k * e / rce * laplaciano_2d(n, n, dx, dx, "neumann") - h_conv / rce * sp.identity(n * n)).tocsr()
    b = (q + h_conv * T_a) / rce
    return A, b, xc

def regime(n, k, rho, c, e):
    A, b, xc = monta_placa(n, k, rho, c, e)
    return spsolve(A.tocsc(), -b).reshape(n, n), xc

T_reg, xc80 = regime(80, e=0.003, **materiais["cobre"])
print(f"cobre, 3 mm, em regime permanente: máxima {T_reg.max():.2f} °C, mínima (nos cantos) {T_reg.min():.2f} °C")

In [ ]:
A80, b80, _ = monta_placa(80, e=0.003, **materiais["cobre"])
sol_ref = solve_ivp(lambda t, T: A80 @ T + b80, (0, 120), np.full(6400, T_a), method="BDF", jac=A80,
                    rtol=1e-6, atol=1e-6, dense_output=True)
fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58], horizontal_spacing=0.2,
                    subplot_titles=["regime permanente (°C)", "temperatura máxima depois de ligar o chip"])
fig.add_trace(go.Heatmap(z=T_reg.T, x=xc80 * 100, y=xc80 * 100, colorscale="Inferno", colorbar=dict(title="°C", x=0.36)), 1, 1)
tt = np.linspace(0, 120, 400)
fig.add_trace(go.Scatter(x=tt, y=sol_ref.sol(tt).max(axis=0), mode="lines", line=dict(color=CORES["aproximacao"], width=3),
                         name="T máxima"), 1, 2)
fig.add_trace(go.Scatter(x=[0, 120], y=[90, 90], mode="lines", line=dict(color=CORES["erro"], dash="dash"), name="limite"), 1, 2)
fig.add_vline(x=30, line=dict(color=CORES["referencia"], dash="dot"), row=1, col=2)
fig.update_xaxes(title="x (cm)", row=1, col=1, scaleanchor="y", constrain="domain")
fig.update_yaxes(title="y (cm)", row=1, col=1)
fig.update_xaxes(title="tempo (s)", row=1, col=2)
fig.update_yaxes(title="temperatura (°C)", range=[25, 95], row=1, col=2)
fig.update_layout(width=980, height=440, showlegend=False, margin=dict(t=110),
                  title="Espalhador de cobre de 3 mm: o calor do chip se espalha e a placa se aquece em dezenas de segundos")
mostrar(fig)

**O que notar:** em regime permanente, a placa sob o chip fica a 88 °C, perto do limite, e os cantos, uns 22 K mais frios. A temperatura sobe depressa nos primeiros segundos e se aproxima do regime em um a dois minutos; a constante de tempo da placa como um todo é $\rho c\,e/h \approx 10$ s. Como calcular essas curvas com um custo que permita testar muitas placas é o assunto do capítulo.

## 2. A ideia em uma imagem

No capítulo 15, discretizamos espaço e tempo juntos. O **método das linhas** separa as duas coisas: discretiza só o espaço e deixa o tempo contínuo. Cada nó da malha ganha uma temperatura $u_i(t)$, uma função do tempo, e a EDP vira um **sistema de EDOs**, uma para cada nó, acopladas aos vizinhos. O nome vem da imagem no plano $x$-$t$: em vez de uma grade de pontos, um feixe de **linhas** verticais, uma por nó, ao longo das quais um integrador de EDOs caminha. Arraste o slider para avançar no tempo numa barra aquecida no meio e com as pontas a 0.

In [ ]:
n_b = 10
x_b = np.linspace(0, 1, n_b + 1)
A_b = laplaciano_1d(n_b - 1, 1 / n_b)
u_b0 = np.where((x_b[1:-1] > 0.3) & (x_b[1:-1] < 0.7), 1.0, 0.0)
sol_b = solve_ivp(lambda t, u: A_b @ u, (0, 0.2), u_b0, method="BDF", jac=A_b, rtol=1e-8, atol=1e-10, dense_output=True)
t_l = np.linspace(0, 0.2, 300)
U_l = sol_b.sol(t_l)
paleta = [CORES["aproximacao"], CORES["secundaria"], CORES["terciaria"], CORES["referencia"], CORES["destaque"]]

def quadro_linhas(t):
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                        subplot_titles=["cada nó é uma EDO: u<sub>i</sub>(t)", "o perfil no instante t"])
    for i in range(n_b - 1):
        cor = paleta[min(i, n_b - 2 - i) % len(paleta)]
        fig.add_trace(go.Scatter(x=t_l, y=U_l[i], mode="lines", line=dict(color=cor, width=2), showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=[t, t], y=[-0.05, 1.05], mode="lines", line=dict(color=CORES["erro"], dash="dash"),
                             showlegend=False), 1, 1)
    u_t = np.r_[0, sol_b.sol(t), 0]
    for i in range(1, n_b):
        cor = paleta[min(i - 1, n_b - 1 - i) % len(paleta)]
        fig.add_trace(go.Scatter(x=[x_b[i], x_b[i]], y=[0, u_t[i]], mode="lines+markers", showlegend=False,
                                 line=dict(color=cor, width=3), marker=dict(size=[0, 11], color=cor)), 1, 2)
    fig.add_trace(go.Scatter(x=x_b, y=u_t, mode="lines", line=dict(color=CORES["exata"], width=1, dash="dot"),
                             showlegend=False), 1, 2)
    fig.update_xaxes(title="tempo t", row=1, col=1)
    fig.update_yaxes(title="u", range=[-0.05, 1.05], row=1, col=1)
    fig.update_xaxes(title="posição x", range=[-0.02, 1.02], row=1, col=2)
    fig.update_yaxes(range=[-0.05, 1.05], row=1, col=2)
    fig.update_layout(width=980, height=420, margin=dict(t=110), title=f"t = {numero(t, 3)}")
    return fig

quadros = [(numero(t, 3), quadro_linhas(t)) for t in [0.0, 0.005, 0.02, 0.05, 0.1, 0.2]]
mostrar(com_controle(quadros, rotulo="t = ", ativo=2, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** cada curva à esquerda é a temperatura de um nó (os nós simétricos têm a mesma cor e a mesma curva). Os nós aquecidos esfriam, os vizinhos esquentam e depois esfriam, todos em direção a zero. O perfil à direita é um corte vertical do feixe de linhas. Para o integrador de EDOs, não há "espaço": há só um vetor $\mathbf u(t)$ com 9 componentes e uma função que calcula $d\mathbf u/dt$. Toda a teoria de estabilidade e de rigidez do capítulo 13 se aplica, e é ela que decide o custo.

## 3. Formulação matemática

### 3.1 O sistema semidiscreto

Para $u_t = \alpha\,u_{xx}$ em $[0, L]$, com $u = 0$ nas pontas e $n$ subintervalos, a diferença centrada nos nós interiores dá

$$
\frac{du_i}{dt} = \frac{\alpha}{\Delta x^2}\,(u_{i+1} - 2u_i + u_{i-1}), \quad i = 1, \dots, n-1
\qquad\Longleftrightarrow\qquad
\frac{d\mathbf u}{dt} = A\,\mathbf u + \mathbf b,
$$ (eq-semidiscreto)

com $A = \frac{\alpha}{\Delta x^2}\,\mathrm{tridiag}(1, -2, 1)$ e $\mathbf b$ com as condições de contorno e as fontes. Os esquemas do capítulo 15 são este sistema integrado por métodos de EDO: FTCS é Euler explícito, BTCS é Euler implícito, Crank-Nicolson é o trapézio. O método das linhas troca a escolha de um esquema "espaço-tempo" pela escolha de **uma discretização espacial** e de **um integrador**, e permite usar integradores adaptativos, de ordem alta e com controle de erro, como os do `solve_ivp`.

Em 2D, com o estêncil de 5 pontos (ou volumes finitos, como no espalhador), $A$ tem $N = n^2$ linhas, com até 5 não nulos em cada uma. A função `laplaciano_2d` do pacote monta $A$ como soma de produtos de Kronecker de duas matrizes 1D.

### 3.2 O espectro e a rigidez

Os autovalores de $A$ em 1D são conhecidos (exercício 1):

$$
\lambda_j = -\frac{4\alpha}{\Delta x^2}\,\sin^2\frac{j\pi}{2n}, \qquad j = 1, \dots, n-1.
$$

- O mais lento, $\lambda_1 \approx -\alpha\pi^2/L^2$, é o modo físico de decaimento: não depende da malha.
- O mais rápido, $\lambda_{n-1} \approx -4\alpha/\Delta x^2$, é um modo que alterna de sinal de um nó para o outro: existe por causa da malha e, numa solução suave, tem amplitude desprezível.

A razão de rigidez, $|\lambda_{n-1}/\lambda_1| \approx (2n/\pi)^2$, **cresce com o quadrado** do número de nós. Em 2D, $|\lambda_{\max}| \approx 8\alpha/\Delta x^2$.

Um método explícito precisa de $h\,|\lambda_{\max}|$ dentro da sua região de estabilidade (capítulo 13): $h \le 2/|\lambda_{\max}| = \Delta x^2/(2\alpha)$ para Euler, que é a condição $r \le 1/2$ do FTCS; $h \lesssim 3{,}3/|\lambda_{\max}|$ para o par de Dormand-Prince do `RK45`. O passo é limitado pela **malha**, não pela precisão, durante toda a simulação. Em 2D, o número de passos cresce como $n^2$ e o custo de cada passo como $n^2$: o custo total cresce como $n^4$. Dobrar a resolução custa 16 vezes mais.

Um método implícito L-estável (BDF, Radau) não tem essa restrição: o passo é escolhido pela precisão com que se quer acompanhar os modos lentos, e os modos rápidos são amortecidos.

### 3.3 A jacobiana: densa, esparsa ou só o padrão

Um método implícito resolve, a cada passo, um sistema não linear com Newton, que precisa da jacobiana $J = \partial\mathbf f/\partial\mathbf u$ (no caso linear, $J = A$). Há três maneiras de fornecê-la ao `solve_ivp`:

| Opção | O que o integrador faz | Custo para $N$ incógnitas |
| --- | --- | --- |
| nada | estima $J$ por diferenças finitas, coluna por coluna, e a guarda **densa** | $N$ avaliações de $\mathbf f$; $8N^2$ bytes; fatoração LU $\sim N^3/3$ operações |
| `jac_sparsity=padrão` | estima $J$ por diferenças finitas, agrupando colunas que não compartilham linhas | poucas avaliações de $\mathbf f$ (5 a 9 em 2D); matriz e LU esparsas |
| `jac=A` (matriz esparsa) | usa a jacobiana exata | nenhuma avaliação extra; matriz e LU esparsas |

O agrupamento de colunas (coloração de Curtis, Powell e Reid) usa o fato de que perturbar $u_j$ só muda as componentes de $\mathbf f$ dos vizinhos de $j$: colunas sem vizinhos em comum podem ser perturbadas **ao mesmo tempo**. Com $N = 6400$ (malha de 80 × 80), a matriz densa tem 41 milhões de entradas (330 MB), enquanto a esparsa tem 32 mil.

### 3.4 Dois erros independentes

A solução do método das linhas tem dois erros que se somam:

$$
\mathbf u_{\text{calculado}} - u_{\text{exato}} = \underbrace{(\mathbf u_{\text{calculado}} - \mathbf u_{\text{semidiscreto}})}_{\text{tempo: controlado por rtol, atol}} + \underbrace{(\mathbf u_{\text{semidiscreto}} - u_{\text{exato}})}_{\text{espaço: } O(\Delta x^2)}.
$$

O integrador adaptativo controla só o primeiro; ele não sabe que as EDOs vieram de uma EDP. Apertar `rtol` além do erro espacial não melhora nada, e refinar a malha com `rtol` frouxo também não. A verificação tem de ser feita **separadamente**: uma malha fixa com tolerâncias decrescentes, e uma tolerância apertada com malhas decrescentes.

### 3.5 Elementos finitos em uma dimensão

Diferenças finitas aproximam a **equação** ponto a ponto. Os elementos finitos aproximam a **solução** por uma combinação de funções simples, e exigem que a equação seja satisfeita em média. Para

$$
-\big(k(x)\,u'\big)' = f(x) \ \text{em}\ (0, L), \qquad u(0) = u_0, \qquad k\,u'(L) = q,
$$

multiplicamos por uma **função teste** $v$ com $v(0) = 0$ e integramos por partes:

$$
\int_0^L k\,u'\,v'\,dx = \int_0^L f\,v\,dx + q\,v(L) \qquad \text{para toda } v.
$$ (eq-forma-fraca)

Essa é a **forma fraca**. Ela só pede derivadas primeiras, e por isso aceita $k$ descontínuo (paredes compostas, materiais diferentes) sem cuidado especial. A condição de fluxo em $x = L$ entrou **naturalmente** no lado direito; só a condição de valor em $x = 0$ precisa ser imposta.

**Galerkin.** Dividimos $[0, L]$ em elementos com nós $x_0 < x_1 < \dots < x_N$ e usamos as **funções chapéu** $\varphi_i$ (lineares por partes, 1 no nó $i$ e 0 nos outros). Escrevendo $u_h = \sum_j U_j \varphi_j$ e tomando $v = \varphi_i$, a forma fraca vira o sistema linear

$$
K\,\mathbf U = \mathbf F, \qquad K_{ij} = \int k\,\varphi_i'\,\varphi_j'\,dx, \qquad F_i = \int f\,\varphi_i\,dx\ (+\,q \text{ no último nó}).
$$

**Montagem.** Cada elemento $[x_e, x_{e+1}]$, de comprimento $h_e$ e condutividade $k_e$, contribui com a **matriz de rigidez do elemento** $\frac{k_e}{h_e}\begin{pmatrix} 1 & -1 \\ -1 & 1\end{pmatrix}$, somada nas linhas e colunas dos seus dois nós. $K$ sai tridiagonal, simétrica e positiva definida (depois de impor $u(0) = u_0$). Em 1D, com $k$ constante por elemento e as integrais de carga calculadas com precisão, a solução de elementos finitos lineares é **exata nos nós**, qualquer que seja a malha. O mesmo procedimento, com triângulos ou tetraedros em vez de segmentos, é o que resolve problemas em geometrias complicadas, e é a razão de ser do método: a forma fraca e a montagem não ligam para o formato do domínio.

### 3.6 Bibliotecas

Escrever o laplaciano à mão, como neste capítulo, é bom para entender e para problemas em retângulos. Para problemas reais, existem bibliotecas que fazem a discretização a partir de uma descrição de alto nível:

- **py-pde**: método das linhas em malhas regulares (cartesianas, cilíndricas, esféricas), com a EDP escrita como texto, por exemplo `"laplace(c) - v * d_dx(c)"`, e integradores explícitos e implícitos;
- **scikit-fem**: elementos finitos em Python puro sobre NumPy e SciPy, leve e fácil de inspecionar;
- **FEniCSx**: elementos finitos em geometrias complexas, com a forma fraca {eq}`eq-forma-fraca` escrita quase como no papel, em linguagem própria (UFL), e solvers paralelos;
- pacotes comerciais de simulação multifísica, que usam os mesmos métodos, com interface gráfica.

Um agente escreve com facilidade o código para qualquer uma delas. Os problemas deste capítulo, de escolha do integrador, de jacobiana e de verificação, continuam existindo dentro de todas.

## 4. À mão

**O espectro de uma malha pequena.** Para $u_t = u_{xx}$ em $[0, 1]$, com 4 subintervalos ($\Delta x = 1/4$, três nós interiores):

$$
A = 16\begin{pmatrix} -2 & 1 & 0 \\ 1 & -2 & 1 \\ 0 & 1 & -2 \end{pmatrix}, \qquad
\lambda = -16\,(2 - \sqrt2),\ -32,\ -16\,(2 + \sqrt 2) \approx -9{,}37;\ -32;\ -54{,}6.
$$

O autovalor mais lento, $-9{,}37$, já está perto do físico, $-\pi^2 \approx -9{,}87$. Euler explícito é estável para $h \le 2/54{,}6 = 0{,}0366$, ou seja, $r = h/\Delta x^2 \le 0{,}586$. É um pouco mais que 1/2, porque numa malha tão grossa o modo mais rápido ainda não chegou a $-4/\Delta x^2 = -64$.

**Elementos finitos com dois elementos.** Para $-u'' = 1$, $u(0) = u(1) = 0$, com dois elementos de $h = 1/2$: só o nó do meio é incógnita. Ele recebe a rigidez dos dois elementos, $K_{11} = 1/h + 1/h = 4$, e a carga $F_1 = \int f\varphi_1\,dx = h = 1/2$ (a área do chapéu). Logo $U_1 = 1/8 = 0{,}125$, exatamente $u(1/2)$ para a solução $u = x(1 - x)/2$.

**Uma parede composta.** Agora $k = 1$ em $[0, 1/2]$ e $k = 10$ em $[1/2, 1]$, sem fonte, com $u(0) = 1$ e $u(1) = 0$. A equação do nó do meio, depois de passar $u(0) = 1$ para o lado direito: $(1/h + 10/h)\,U_1 = (1/h)\cdot 1$, ou $U_1 = 1/11 \approx 0{,}0909$. É o resultado das resistências térmicas em série: $R_1 = 0{,}5/1$, $R_2 = 0{,}5/10$, e a queda de temperatura na primeira camada é $R_1/(R_1 + R_2) = 10/11$ do total.

In [ ]:
print("autovalores de A (n = 4):", np.round(np.linalg.eigvalsh(laplaciano_1d(3, 0.25).toarray()), 3))
print("FEM, -u'' = 1:", fem_1d(1.0, 1.0, [0, 0.5, 1], 0.0, ("dirichlet", 0.0)).u)
print("FEM, parede composta:", np.round(fem_1d(lambda x: np.where(x < 0.5, 1.0, 10.0), 0.0, [0, 0.5, 1], 1.0,
                                               ("dirichlet", 0.0)).u, 4))

## 5. Explore

### 5.1 O passo do RK45 é decidido pela malha

A mesma barra da seção 2, integrada até $t = 0{,}5$ com `RK45` e com `BDF` (com a jacobiana esparsa), para malhas cada vez mais finas. À esquerda, os autovalores do sistema; à direita, o passo que cada integrador escolheu ao longo do tempo.

In [ ]:
def quadro_passos(n):
    dx = 1 / n
    xi = np.linspace(0, 1, n + 1)[1:-1]
    A = laplaciano_1d(n - 1, dx)
    u0 = np.where((xi > 0.3) & (xi < 0.7), 1.0, 0.0)
    lam = np.linalg.eigvalsh(A.toarray())
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                        subplot_titles=["autovalores |λ| do sistema", "passo de tempo escolhido"])
    fig.add_trace(go.Scatter(x=np.arange(1, n), y=np.sort(np.abs(lam)), mode="markers", showlegend=False,
                             marker=dict(color=CORES["aproximacao"], size=6)), 1, 1)
    fig.add_trace(go.Scatter(x=[1, n - 1], y=[np.pi**2] * 2, mode="lines", line=dict(color=CORES["exata"], dash="dash"),
                             showlegend=False), 1, 1)
    fig.add_annotation(x=np.log10(n - 1), y=np.log10(np.pi**2), text="modo físico mais lento, π²", showarrow=False,
                       xanchor="right", yshift=-12, row=1, col=1)
    passos = {}
    for met, kw, cor in [("RK45", {}, CORES["erro"]), ("BDF", {"jac": A}, CORES["aproximacao"])]:
        s = solve_ivp(lambda t, u: A @ u, (0, 0.5), u0, method=met, **kw)
        passos[met] = len(s.t) - 1
        fig.add_trace(go.Scatter(x=s.t[1:], y=np.diff(s.t), mode="lines+markers", marker=dict(size=4), name=met,
                                 line=dict(color=cor)), 1, 2)
    fig.add_trace(go.Scatter(x=[0, 0.5], y=[3.3 / abs(lam).max()] * 2, mode="lines", name="3,3/|λ<sub>max</sub>|",
                             line=dict(color=CORES["referencia"], dash="dot")), 1, 2)
    fig.update_xaxes(title="índice j", type="log", row=1, col=1)
    fig.update_yaxes(title="|λ<sub>j</sub>|", type="log", range=[0.5, 5.5], exponentformat="power", row=1, col=1)
    fig.update_xaxes(title="tempo t", row=1, col=2)
    fig.update_yaxes(title="passo h", type="log", range=[-5, -0.5], exponentformat="power", row=1, col=2)
    fig.update_layout(width=980, height=430, margin=dict(t=110), legend=dict(orientation="h", y=-0.22),
                      title=f"n = {n}: RK45 dá {passos['RK45']} passos; BDF, {passos['BDF']}")
    return fig

quadros = [(str(n), quadro_passos(n)) for n in [10, 20, 40, 80]]
mostrar(com_controle(quadros, rotulo="subintervalos n = ", ativo=2, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** a cada vez que $n$ dobra, o maior autovalor quadruplica e o passo do `RK45` cai a um quarto, colado na linha $3{,}3/|\lambda_{\max}|$ (a fronteira da sua região de estabilidade) do começo ao fim, mesmo quando a solução já está quase parada. O `BDF` começa com passos pequenos, para acompanhar o transiente inicial, e depois os aumenta mais de mil vezes; o número de passos quase não muda com a malha.

### 5.2 O aquecimento da placa e o projeto

O campo de temperatura do espalhador de cobre de 3 mm, nos primeiros segundos depois de ligar o chip:

In [ ]:
def quadro_campo(t):
    T = sol_ref.sol(t).reshape(80, 80)
    fig = go.Figure(go.Heatmap(z=T.T, x=xc80 * 100, y=xc80 * 100, zmin=30, zmax=90, colorscale="Inferno",
                               colorbar=dict(title="°C")))
    fig.update_xaxes(title="x (cm)", scaleanchor="y", constrain="domain")
    fig.update_yaxes(title="y (cm)")
    fig.update_layout(width=620, height=520, title=f"t = {numero(t, 1)} s: máxima {numero(T.max(), 1)} °C, "
                                                   f"mínima {numero(T.min(), 1)} °C")
    return fig

quadros = [(numero(t, 1), quadro_campo(t)) for t in [0.5, 2.0, 5.0, 10.0, 30.0, 120.0]]
mostrar(com_controle(quadros, rotulo="t (s) = ", ativo=2, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** no primeiro segundo, só a região sob o chip se aquece: o calor ainda não teve tempo de se espalhar (o comprimento de difusão $\sqrt{\alpha t}$ é de uns 3 mm em 0,1 s e 1 cm em 1 s). Depois, a placa inteira sobe junta, com a forma do campo quase fixa: é o modo lento, de constante de tempo $\rho c\,e/h$.

Para o projeto, uma observação poupa quase todo o trabalho: a equação é linear e a fonte é constante, então a temperatura sobe **monotonamente** até o regime permanente (princípio de comparação). O pior caso é o regime, que é um problema **elíptico**: basta resolver $A\mathbf T = -\mathbf b$ uma vez, sem integrar no tempo.

In [ ]:
print("material   espessura   T máx. (°C)   massa (g)")
for nome, mat in materiais.items():
    for e_mm in [1, 2, 3, 4, 5]:
        T, _ = regime(80, e=e_mm / 1000, **mat)
        massa = mat["rho"] * L_p**2 * e_mm / 1000 * 1000
        marca = "  <- atende" if T.max() < 90 else ""
        print(f"{nome:9s} {e_mm:6d} mm {T.max():12.1f} {massa:11.1f}{marca}")

A placa de alumínio de 5 mm atende ao limite com metade da massa da de cobre de 3 mm: o alumínio conduz 40 % menos, mas é mais de três vezes mais leve. Espessura e condutividade entram quase só pelo produto $k\,e$, a **condutância** da placa (repare que o alumínio de 5 mm e o cobre de 3 mm têm $k\,e$ parecidos).

### 5.3 Elementos finitos numa placa de combustível

Uma placa de combustível nuclear tem um núcleo de óxido de urânio ($k_f = 3$ W/(m K)) com geração de calor $q''' = 10^8$ W/m³ e meia-espessura de 5 mm, revestido por 1 mm de liga metálica ($k_r = 15$ W/(m K)) cuja superfície está a 300 °C. Pela simetria, basta a metade: $x = 0$ na superfície, $x = 6$ mm no centro, onde o fluxo é zero. A temperatura no centro é a grandeza de projeto. Mude o número de elementos (iguais) e compare com a solução exata, que é linear no revestimento e parabólica no núcleo.

In [ ]:
q_g, k_f, k_r, a_f, e_r, T_s = 1e8, 3.0, 15.0, 0.005, 0.001, 300.0
L_c = a_f + e_r
k_comb = lambda x: np.where(x < e_r, k_r, k_f)
f_comb = lambda x: np.where(x < e_r, 0.0, q_g)

def T_comb(x):
    return np.where(x < e_r, T_s + q_g * a_f * x / k_r,
                    T_s + q_g * a_f * e_r / k_r + q_g / (2 * k_f) * (a_f**2 - (L_c - x) ** 2))

xx = np.linspace(0, L_c, 400)

def quadro_fem(n):
    nos = np.linspace(0, L_c, n + 1)
    r = fem_1d(k_comb, f_comb, nos, T_s, ("neumann", 0.0))
    fig = go.Figure()
    fig.add_vrect(x0=0, x1=e_r * 1000, fillcolor=CORES["referencia"], opacity=0.15, line_width=0)
    fig.add_annotation(x=e_r * 500, y=820, text="revestimento", showarrow=False)
    fig.add_annotation(x=3.5, y=820, text="combustível (com geração)", showarrow=False)
    fig.add_trace(go.Scatter(x=xx * 1000, y=T_comb(xx), mode="lines", name="exata", line=dict(color=CORES["exata"], dash="dash")))
    fig.add_trace(go.Scatter(x=nos * 1000, y=r.u, mode="lines+markers", name="elementos finitos",
                             line=dict(color=CORES["aproximacao"], width=3), marker=dict(size=9)))
    erro_c = round(float(r.u[-1] - T_comb(L_c)), 1) + 0.0
    no_interface = np.any(np.isclose(nos, e_r))
    fig.update_xaxes(title="distância à superfície (mm)")
    fig.update_yaxes(title="temperatura (°C)", range=[280, 900])
    fig.update_layout(width=780, height=440, legend=dict(x=0.02, y=0.75),
                      title=f"{n} elementos ({'com' if no_interface else 'sem'} nó na interface): "
                            f"erro no centro {numero(erro_c, 1)} K")
    return fig

quadros = [(str(n), quadro_fem(n)) for n in [2, 3, 4, 6, 12]]
mostrar(com_controle(quadros, rotulo="elementos: ", ativo=3, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com 6 ou 12 elementos, há um nó exatamente na interface entre os materiais, e os valores nodais coincidem com a solução exata (a menos do arredondamento), como prevê a seção 3.5; entre os nós, a aproximação é a interpolação linear. Com 2, 3 ou 4 elementos, um elemento atravessa a interface e recebe uma condutividade só (a do seu ponto médio), e o centro erra por dezenas ou centenas de kelvins. A regra prática, que vale para diferenças finitas também (capítulo 14, seção 7.2): **a malha deve respeitar as interfaces entre materiais**. Os geradores de malha das bibliotecas de elementos finitos fazem isso a partir da geometria.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Simule em Python o aquecimento do espalhador de cobre de 3 mm (equação do calor 2D com fonte do chip e perda para a placa fria) nos primeiros 30 s depois de ligar o chip, numa malha de 80 × 80. Depois, quero repetir a simulação para umas 200 combinações de material e espessura para escolher a placa mais leve que fique abaixo de 90 °C."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
import time
import numpy as np
from scipy.integrate import solve_ivp

k, rho, cp, e = 390.0, 8900.0, 385.0, 0.003       # cobre, espessura 3 mm
Lp, n = 0.05, 80
dx = Lp / n
alpha = k / (rho * cp)
xc = (np.arange(n) + 0.5) * dx
X, Y = np.meshgrid(xc, xc, indexing="ij")
chip = (np.abs(X - Lp / 2) < 0.005) & (np.abs(Y - Lp / 2) < 0.005)
q = np.where(chip, 100.0 / 0.01**2, 0.0)          # W/m² sob o chip
h, Ta = 1000.0, 30.0

def rhs(t, Tvec):
    T = Tvec.reshape(n, n)
    Tp = np.pad(T, 1, mode="edge")                   # bordas isoladas (fluxo zero)
    lap = (Tp[2:, 1:-1] + Tp[:-2, 1:-1] + Tp[1:-1, 2:] + Tp[1:-1, :-2] - 4 * T) / dx**2
    dTdt = alpha * lap + (q - h * (T - Ta)) / (rho * cp * e)
    return dTdt.ravel()

T0 = np.full(n * n, Ta)
t0 = time.time()
sol = solve_ivp(rhs, (0, 30), T0, method="RK45")
duracao = time.time() - t0
print(f"T máxima em t = 30 s: {sol.y[:, -1].max():.2f} °C  ({len(sol.t) - 1} passos, {sol.nfev} avaliações, {duracao:.1f} s)")
print(f"estimativa para 200 casos: {200 * duracao / 60:.0f} min")

O código está correto: o `np.pad(..., mode="edge")` reproduz a condição de borda isolada nos volumes finitos, e a temperatura máxima bate com a curva da seção 1. O agente comenta o custo assim: *"o passo de tempo é pequeno porque o aquecimento no início é muito rápido; para os 200 casos, sugiro reduzir a malha para 20 × 20 ou rodar em paralelo"*. As duas sugestões tratam o sintoma. Qual é a causa?

### 6.2 Verificação

**Balanço de energia.** A energia entregue pelo chip em 30 s tem de estar armazenada na placa ou ter saído para o fluido: $P\,t = \int \rho c\,e\,(T - T_a)\,dA + \int_0^t\!\!\int h\,(T - T_a)\,dA\,dt'$.

In [ ]:
dA = dx * dx
armazenada = rho * cp * e * np.sum(sol.y[:, -1] - Ta) * dA
perdida = trapezoid(h * np.sum(sol.y - Ta, axis=0) * dA, sol.t)
print(f"entregue: {100.0 * 30:.1f} J;  armazenada: {armazenada:.1f} J;  perdida: {perdida:.1f} J;  "
      f"soma: {armazenada + perdida:.1f} J")

O balanço fecha: a física da discretização está certa.

**O passo ao longo do tempo.** Se o passo fosse pequeno por causa do transiente inicial, ele cresceria quando a placa se aproxima do regime:

In [ ]:
passos = np.diff(sol.t)
for t_ in [0.1, 1, 5, 15, 29]:
    i = np.searchsorted(sol.t, t_)
    print(f"t ≈ {t_:4} s: passo do RK45 = {passos[i]:.2e} s")

O passo é praticamente o mesmo do começo ao fim, cerca de 1,4 ms, inclusive no fim, quando a temperatura máxima muda menos de 0,1 K por segundo. O diagnóstico do agente está errado: o passo não é limitado pela precisão.

**Erro de tempo e erro de espaço, separados.** Para medir cada erro, precisamos de uma referência no tempo (a mesma malha, tolerância apertada) e de uma sequência de malhas (tolerância apertada). Com `RK45`, cada uma dessas simulações custaria o mesmo que a do agente ou mais; adiantando a correção da seção 6.4, usamos `BDF` com a jacobiana esparsa:

In [ ]:
def T_max_30(n_, rtol=1e-8):
    A, b, _ = monta_placa(n_, e=0.003, **materiais["cobre"])
    s = solve_ivp(lambda t, T: A @ T + b, (0, 30), np.full(n_ * n_, T_a), method="BDF", jac=A, rtol=rtol, atol=1e-8)
    return s.y[:, -1].max()

ref_tempo = T_max_30(80)
print(f"erro de tempo do RK45 (rtol padrão 1e-3), malha 80 x 80: {sol.y[:, -1].max() - ref_tempo:+.3f} K")
print("erro de espaço (tolerância apertada):")
vals = {n_: T_max_30(n_) for n_ in [10, 20, 40, 80, 160]}
for n_, v in vals.items():
    print(f"   {n_:3d} x {n_:<3d}: T máx = {v:.3f} °C, diferença para a malha 160: {v - vals[160]:+.3f} K")

Os dois erros são pequenos, de milésimos de kelvin com a malha de 80 × 80, e a malha de 20 × 20 sugerida pelo agente erraria por uns 0,1 K. A simulação do agente está **certa**. O problema é outro: o custo.

### 6.3 Por quê

O sistema semidiscreto é $d\mathbf T/dt = A\,\mathbf T + \mathbf b$. Os seus autovalores extremos (seção 3.2):

In [ ]:
rce = rho * cp * e
lam_lento = h / rce                                    # placa inteira esfriando junto
lam_rapido = 8 * alpha / dx**2 + h / rce               # modo xadrez, de célula para célula
print(f"modo mais lento: |λ| = {lam_lento:.4f} 1/s (constante de tempo {1 / lam_lento:.1f} s)")
print(f"modo mais rápido: |λ| = {lam_rapido:.0f} 1/s (constante de tempo {1e3 / lam_rapido:.2f} ms)")
print(f"razão de rigidez: {lam_rapido / lam_lento:,.0f}".replace(",", "."))
passos_30 = f"{30 * lam_rapido / 3.3:,.0f}".replace(",", ".")
print(f"passo máximo estável do RK45 (3,3/|λ|): {3.3 / lam_rapido * 1e3:.2f} ms  ->  {passos_30} passos em 30 s")

In [ ]:
n_m = 20
dx_m = L_p / n_m
xm = (np.arange(n_m) + 0.5) * dx_m
modo_lento = np.ones((n_m, n_m))
modo_rapido = np.cos(np.pi * np.arange(n_m))[:, None] * np.cos(np.pi * np.arange(n_m))[None, :]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=[f"modo mais lento: constante de tempo {numero(1 / lam_lento, 1)} s",
                                    f"modo mais rápido: constante de tempo {numero(1e3 / lam_rapido, 2)} ms"])
for col, M in [(1, modo_lento), (2, modo_rapido)]:
    fig.add_trace(go.Heatmap(z=M.T, x=xm * 100, y=xm * 100, zmin=-1, zmax=1, colorscale="RdBu", showscale=(col == 2)), 1, col)
    fig.update_xaxes(title="x (cm)", row=1, col=col, scaleanchor=f"y{'' if col == 1 else col}", constrain="domain")
fig.update_yaxes(title="y (cm)", row=1, col=1)
fig.update_layout(width=900, height=440, margin=dict(t=110), title="Os modos extremos do sistema semidiscreto (malha de 20 × 20 para enxergar)")
mostrar(fig)

O modo mais rápido é um **xadrez**: cada célula com sinal oposto ao das vizinhas. Ele não existe na física da placa (é um artefato da malha) e, na solução, tem amplitude zero. Mas um método explícito precisa manter $h\,\lambda$ na sua região de estabilidade para **todos** os modos, inclusive os que não aparecem: o `RK45` dá 21 mil passos de 1,4 ms para acompanhar uma solução cuja escala de tempo é de 10 s. A razão de rigidez de 24 mil cresce como $n^2$: na malha de 160 × 160, seriam 85 mil passos, cada um 4 vezes mais caro. As sugestões do agente (malha mais grossa, paralelismo) atacam o custo de cada passo, não o número de passos.

### 6.4 A correção

Um método implícito L-estável, com a jacobiana esparsa. Aqui a jacobiana é a própria matriz $A$ (o problema é linear); num problema não linear, `jac_sparsity` com o padrão de $A$ dá o mesmo ganho:

In [ ]:
A, b, _ = monta_placa(80, e=0.003, **materiais["cobre"])
for metodo, opcoes in [("BDF", {"jac": A}), ("BDF", {"jac_sparsity": A != 0}), ("Radau", {"jac": A})]:
    t0 = time.time()
    s = solve_ivp(lambda t, T: A @ T + b, (0, 30), np.full(6400, T_a), method=metodo, **opcoes)
    dur = time.time() - t0
    print(f"{metodo:5s} com {list(opcoes)[0]:12s}: T máx = {s.y[:, -1].max():.3f} °C, {len(s.t) - 1:3d} passos, "
          f"{s.nfev:4d} avaliações, {dur:.2f} s ({duracao / dur:.0f} vezes mais rápido que o RK45)")

O `BDF` dá umas 500 vezes menos passos de tempo, e a simulação fica dezenas de vezes mais rápida (os tempos dependem do computador; as razões, pouco). Com a malha de 160 × 160, a vantagem cresce para centenas de vezes. Os 200 casos passam a levar menos de um minuto, sem engrossar a malha.

E, como vimos na seção 5.2, se a pergunta é só a temperatura máxima, nem é preciso integrar no tempo: o regime permanente é uma única solução esparsa por caso. A resposta do projeto: **alumínio de 5 mm** (34 g), em vez de cobre de 3 mm (67 g).

> **Lição.** O método das linhas produz sistemas **rígidos**, com rigidez que cresce como $1/\Delta x^2$, porque a malha cria modos rápidos que a física não tem. Um integrador explícito funciona, mas com o passo preso à malha, e o custo cresce como $n^4$ em 2D. Peça ao agente um integrador implícito (`BDF`, `Radau`) **com a jacobiana esparsa** ou o seu padrão; verifique o resultado com um **balanço de energia** e com estudos **separados** de tolerância e de malha; e desconfie da explicação "o passo é pequeno porque a solução muda rápido" sem olhar o passo ao longo do tempo.

## 7. Quando falha

### 7.1 BDF sem a jacobiana esparsa

A reação natural a "o problema é rígido" é trocar `method="RK45"` por `method="BDF"` e mais nada. Numa malha de 60 × 60:

In [ ]:
A60, b60, _ = monta_placa(60, e=0.003, **materiais["cobre"])
chamadas = [0]

def f60(t, T):
    chamadas[0] += 1                      # conta todas as chamadas, inclusive as usadas para estimar a jacobiana
    return A60 @ T + b60

for metodo, opcoes, nome in [("RK45", {}, "RK45"), ("BDF", {}, "BDF sem jacobiana"),
                             ("BDF", {"jac_sparsity": A60 != 0}, "BDF com jac_sparsity"), ("BDF", {"jac": A60}, "BDF com jac=A")]:
    chamadas[0] = 0
    t0 = time.time()
    s = solve_ivp(f60, (0, 30), np.full(3600, T_a), method=metodo, **opcoes)
    print(f"{nome:22s}: {time.time() - t0:6.2f} s, {len(s.t) - 1:6d} passos, {chamadas[0]:6d} chamadas de f")
for n_ in [60, 80, 200]:
    print(f"jacobiana densa na malha {n_} x {n_}: {(n_ * n_) ** 2 * 8 / 1e9:.2f} GB")

Sem a jacobiana, o `BDF` dá os mesmos poucos passos, mas fica **mais lento** que o `RK45`: ele estima a jacobiana densa com uma chamada de $\mathbf f$ por coluna (3 600 aqui) e a fatora como matriz cheia, com custo $N^3$. Com `jac_sparsity`, a coloração da seção 3.3 reduz a estimativa a uma dezena de chamadas. Na malha de 80 × 80, a fatoração densa já leva vários segundos e ocupa 330 MB; na de 200 × 200, não cabe na memória de um computador comum. Quem tirar daí a conclusão "o implícito é pior" está errado: o que faltou foi a **esparsidade**.

### 7.2 Apertar a tolerância não refina a malha

A barra da seção 2, agora com solução exata $u = e^{-\pi^2 t}\sin\pi x$, integrada até $t = 0{,}1$ com `BDF` e tolerâncias cada vez menores, para três malhas:

In [ ]:
rtols = np.array([1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8])
erros_malha = {}
for n_ in (10, 40, 160):
    xi = np.linspace(0, 1, n_ + 1)[1:-1]
    A1 = laplaciano_1d(n_ - 1, 1 / n_)
    erros = []
    for rt in rtols:
        s = solve_ivp(lambda t, u: A1 @ u, (0, 0.1), np.sin(np.pi * xi), method="BDF", jac=A1, rtol=rt, atol=rt * 1e-3)
        erros.append(np.max(np.abs(s.y[:, -1] - np.exp(-np.pi**2 * 0.1) * np.sin(np.pi * xi))))
    erros_malha[n_] = erros
    print(f"n = {n_:3d}: erro com rtol = 1e-8: {erros[-1]:.1e}")

In [ ]:
fig = go.Figure()
for n_, cor in [(10, CORES["erro"]), (40, CORES["secundaria"]), (160, CORES["aproximacao"])]:
    fig.add_trace(go.Scatter(x=rtols, y=erros_malha[n_], mode="lines+markers", name=f"n = {n_}", line=dict(color=cor, width=2.5)))
fig.update_xaxes(title="tolerância relativa rtol", type="log", autorange="reversed", exponentformat="power")
fig.update_yaxes(title="erro máximo em t = 0,1", type="log", exponentformat="power")
fig.update_layout(width=780, height=450, title="O erro para de cair quando o erro da malha domina",
                  legend=dict(orientation="h", y=-0.22))
mostrar(fig)

Com 40 e 160 subintervalos, a curva cai enquanto o erro de tempo domina e depois fica **plana**, no nível do erro da malha, $O(\Delta x^2)$: dividir $\Delta x$ por 4 baixa o patamar 16 vezes. Com 10, o erro da malha domina desde o início (na tolerância mais frouxa, os dois erros se cancelam em parte, por acaso). Com 10 subintervalos, apertar `rtol` de $10^{-4}$ para $10^{-8}$ multiplica o custo e não ganha nada. É o erro mais comum ao "verificar" uma simulação de EDP: apertar só a tolerância, ver que a resposta não muda, e concluir que ela está convergida.

### 7.3 O método das linhas não conserta a discretização espacial

O rio do capítulo 15 pelo método das linhas, com $\Delta x = 200$ m e um integrador preciso (`RK45` com `rtol` = $10^{-8}$), para as duas discretizações espaciais da advecção:

In [ ]:
u_r, D_r, dx_r = 0.5, 5.0, 200.0
x_r = np.arange(0, 30_000 + dx_r, dx_r)
N_r = x_r.size
c_rio = lambda x, t: 50 * 150 / np.sqrt(150**2 + 2 * D_r * t) * np.exp(-(x - 2000 - u_r * t) ** 2 / (2 * (150**2 + 2 * D_r * t)))
dif2 = sp.diags([np.ones(N_r - 1), -2 * np.ones(N_r), np.ones(N_r - 1)], [-1, 0, 1]) / dx_r**2
derivadas = {"centrada": sp.diags([-np.ones(N_r - 1), np.ones(N_r - 1)], [-1, 1]) / (2 * dx_r),
             "upwind": sp.diags([-np.ones(N_r - 1), np.ones(N_r)], [-1, 0]) / dx_r}
for nome, Dx in derivadas.items():
    M = (D_r * dif2 - u_r * Dx).tolil()
    M[0, :] = 0
    M[-1, :] = 0
    s = solve_ivp(lambda t, c: M.tocsr() @ c, (0, 36_000), c_rio(x_r, 0), method="RK45", rtol=1e-8, atol=1e-10)
    print(f"advecção {nome:9s}: pico na captação {s.y[100, -1]:5.2f} mg/L, menor concentração {s.y[:, -1].min():6.2f} mg/L "
          f"(exato: {c_rio(20_000, 36_000):.2f} e 0)")

Com o tempo integrado com precisão, restam só os erros espaciais, e eles são os do capítulo 15. A diferença centrada oscila e cria concentrações negativas (o número de Péclet da malha é 20). O upwind fica ainda **pior** que o esquema explícito do agente do capítulo 15 (que dava 5,07 mg/L): a difusão numérica do upwind semidiscreto é $u\,\Delta x/2 = 50$ m²/s, sem o fator $(1 - C)$ que o passo de tempo finito compensava. O método das linhas separa espaço e tempo, mas não resolve o problema de nenhum dos dois: para a advecção, a discretização **espacial** precisa ser de ordem alta e com limitadores, ou a malha precisa ser fina.

## Resumo

| Escolha | Opções | Quando usar | Cuidado |
| --- | --- | --- | --- |
| integrador para difusão | `BDF`, `Radau` (implícitos, L-estáveis) | sempre que a malha é fina | passar `jac` esparsa ou `jac_sparsity` |
| integrador para difusão | `RK45`, `RK23` (explícitos) | malhas grossas, poucos passos | passo $\propto \Delta x^2$; custo $\propto n^4$ em 2D |
| integrador para advecção pura | RK explícitos (SSP-RK) | passo pela condição CFL, $\propto \Delta x$ | espaço com upwind de ordem alta e limitador |
| jacobiana | densa / `jac_sparsity` / `jac` esparsa | sempre a esparsa | densa: memória $8N^2$, LU $N^3$ |
| regime permanente | solução esparsa direta | quando só o estado final importa | não integrar até "parar" |
| elementos finitos 1D | Galerkin com funções chapéu | coeficientes descontínuos, fluxos naturais | malha deve respeitar as interfaces |
| bibliotecas | py-pde, scikit-fem, FEniCSx | geometrias e problemas reais | a escolha do integrador e a verificação continuam suas |

## Exercícios

**Conceituais (sem computador)**

1. Mostre que os vetores $v_i = \sin(ij\pi/n)$ são autovetores da matriz $\mathrm{tridiag}(1, -2, 1)$ de ordem $n - 1$, com autovalores $-4\sin^2(j\pi/(2n))$.
2. Para o espalhador, com volumes finitos e bordas isoladas, mostre que o vetor constante é autovetor de $A$ com autovalor $-h/(\rho c\,e)$. Por que esse é o modo mais lento?
3. Quantas avaliações de $\mathbf f$ a estimativa da jacobiana com `jac_sparsity` precisa, em princípio, para uma matriz tridiagonal? E para o estêncil de 5 pontos em 2D? (Pense em quais colunas podem ser perturbadas juntas.)
4. Monte à mão a matriz de rigidez global de três elementos lineares de comprimentos $h_1, h_2, h_3$ e condutividades $k_1, k_2, k_3$, e mostre que a equação de cada nó interior é o equilíbrio de fluxos entre duas resistências térmicas em série.

**Verificação (com agente)**

5. Peça a um agente a simulação do espalhador com py-pde ou scikit-fem. Compare a temperatura máxima com a deste capítulo, e verifique a ordem da malha pelo estudo da seção 6.2.
6. Peça a um agente a simulação do aquecimento de um bloco de concreto (calor de hidratação) com o método das linhas em 1D, com `solve_ivp`. Meça o número de passos de `RK45` e de `BDF` em função da malha, e confirme as taxas $n^2$ e $\approx$ constante.
7. Peça a um agente para resolver a placa de combustível da seção 5.3 por diferenças finitas, com a condutividade avaliada nos nós. Compare com elementos finitos e com a solução exata, com e sem um nó na interface, e meça a ordem.

**Projeto aberto**

8. **Chip fora do centro e dois chips.** Num módulo real, há dois chips de potências diferentes, perto de uma borda. Modele o espalhador com chips em posições arbitrárias, com o método das linhas e `BDF` com jacobiana esparsa, e use o regime permanente para escolher posições, material e espessura que minimizem a massa com a temperatura de cada chip abaixo de 90 °C. Verifique o resultado final com uma malha mais fina e com o balanço de energia.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 30 e 31.
- W. E. Schiesser, *The Numerical Method of Lines*, Academic Press, 1991.
- L. F. Shampine e M. W. Reichelt, "The MATLAB ODE Suite", *SIAM Journal on Scientific Computing*, 18(1), 1997 (integradores para sistemas rígidos e jacobianas esparsas).
- C. Johnson, *Numerical Solution of Partial Differential Equations by the Finite Element Method*, Dover, 2009.
- Documentação de py-pde, scikit-fem e FEniCSx (tutoriais com a equação do calor).